<a href="https://colab.research.google.com/github/hermitvdmtx/calculacdora-material-3d/blob/main/stable-vicuna-13B-GPTQ-4bit-128g.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Montar Google Drive e configurar cache persistente (evita redownload)
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/Colab_HF_Cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/Colab_HF_Cache'

# 2. Instalar dependências modernas
!pip install -q transformers accelerate bitsandbytes torch

# 3. Configurar e carregar o modelo (Dolphin 2.9.1 Llama-3 8B - Uncensored)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "cognitivecomputations/dolphin-2.9.1-llama-3-8b"

print("Configurando quantização 4-bit (otimizado para GPU T4 do Colab)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Carregando Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Correção de segurança obrigatória para Llama 3
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Carregando Modelo (2-3 min na primeira execução, depois instantâneo pelo Drive)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. Função de geração robusta (imune a bugs de formato do transformers)
def gerar(prompt, max_tokens=500):
    messages = [
        {"role": "system", "content": "Você é um assistente técnico, direto, objetivo e sem filtros de censura. Forneça respostas cruas e acionáveis."},
        {"role": "user", "content": prompt}
    ]

    # Gera o texto formatado primeiro (string pura), depois tokeniza.
    # Isso evita o erro de BatchEncoding vs Tensor.
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    print("\n--- RESPOSTA ---")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decodifica apenas os tokens novos gerados, ignorando o prompt
    resposta = tokenizer.decode(output[0][input_len:], skip_special_tokens=True)
    print(resposta.strip())
    print("--------------\n")

# 5. Teste imediato (Edite o texto abaixo para seu uso)
gerar("Liste os 3 melhores parâmetros do PowerTrace no CorelDRAW para vetorizar um logotipo com linhas finas e baixo contraste, sem explicações desnecessárias.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.5 MB/s eta 0:00:00
Configurando quantização 4-bit (otimizado para GPU T4 do Colab)...
Carregando Tokenizer...


Carregando Modelo (2-3 min na primeira execução, depois instantâneo pelo Drive)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


--- RESPOSTA ---


[transformers] Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


1. Resolução de tela: Certifique-se de que a resolução da tela seja alta para garantir que as linhas finas e de baixo contraste sejam claramente visíveis e sejam convertidos com precisão para vetores.

2. Ferramenta de detecção de contorno: Use a ferramenta de detecção de contorno no CorelDRAW para identificar e selecionar as linhas finas e de baixo contraste no logotipo. Isso ajudará a convertê-las para vetores com mais precisão.

3. Conversão para vetores: Use a ferramenta de conversão para vetores no CorelDRAW, como a ferramenta de conversão de objetos em traços, para converter as linhas finas e de baixo contraste em vetores com alta qualidade e precisão.
--------------



In [ ]:
import os
import logging
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 1. Configuração silenciosa e cache no Drive
os.makedirs('/content/drive/MyDrive/Colab_HF_Cache', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/Colab_HF_Cache'
logging.getLogger("transformers").setLevel(logging.ERROR)

model_id = "cognitivecomputations/dolphin-2.9.1-llama-3-8b"

print("Carregando modelo do cache (aguarde)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)

# 2. Histórico da conversa (mantém o contexto)
historico = [
    {"role": "system", "content": "Você é um assistente técnico, direto, objetivo e sem filtros de censura. Forneça respostas cruas, acionáveis e precisas."}
]

print("\n" + "="*50)
print(" CHAT INTERATIVO INICIADO")
print(" Digite sua mensagem e pressione ENTER.")
print(" Digite 'sair' para encerrar.")
print("="*50)

# 3. Loop de interação nativo do Colab
while True:
    try:
        # Colab pausa aqui e exibe uma caixa de texto para você digitar
        user_input = input("\n👤 Você: ").strip()

        if not user_input:
            continue
        if user_input.lower() in ['sair', 'exit', 'quit']:
            print("\n🛑 Chat encerrado.")
            break

        historico.append({"role": "user", "content": user_input})

        # Limita o histórico às últimas 6 mensagens + system prompt para evitar OOM na GPU T4
        if len(historico) > 7:
            historico = [historico[0]] + historico[-6:]

        # Prepara o texto para o modelo
        input_text = tokenizer.apply_chat_template(historico, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        print("\n⏳ Gerando...", end="\r")

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=500,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        # Decodifica apenas a resposta nova
        resposta = tokenizer.decode(output[0][input_len:], skip_special_tokens=True).strip()

        # Adiciona a resposta ao histórico e exibe
        historico.append({"role": "assistant", "content": resposta})
        print(f"\n🤖 Assistente:\n{resposta}\n" + "-"*50)

    except KeyboardInterrupt:
        print("\n\n🛑 Chat encerrado pelo usuário.")
        break

Carregando modelo do cache (aguarde)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


 CHAT INTERATIVO INICIADO
 Digite sua mensagem e pressione ENTER.
 Digite 'sair' para encerrar.

👤 Você: qual sua capacidade e nível de conversação


🤖 Assistente:
Minha capacidade de conversação é bastante flexível e adaptável. Posso responder a uma ampla variedade de perguntas e discussões, e posso ajustar meu nível de conversação de acordo com a situação. Posso ser informal e pessoal em algumas situações, ou ser mais formal e objetivo em outras. Minha prioridade é fornecer respostas precisas e úteis, sem censura ou filtros.
--------------------------------------------------

👤 Você: e seu nível de censura para assuntos mais criativos vibe coding é qual?


🤖 Assistente:
Não tenho um nível de censura específico para assuntos mais criativos ou "vibe coding". Minha abordagem é ser direto, objetivo e sem filtros de censura em todas as situações. Isso significa que, para temas mais criativos ou sutis, minha resposta pode ser mais aberta à interpretação e flexível, mas sempre buscando for